In [5]:
# =========================================================================
# CREDIT RISK PROJECT
# DATA CLEANING + PREPROCESSING + XGBOOST + COST-SENSITIVE LEARNING
# + SELECTIVE PREDICTION + RELIABILITY MONITORING
# + SAVE COMPLETE DEPLOYMENT PACKAGE AS PKL
#
# NOTE:
# All plots and visualization sections have been removed.
# =========================================================================


# =========================================================================
# 1. IMPORT LIBRARIES AND CREATE OUTPUT DIRECTORY
# =========================================================================

import os
import re
import json
import traceback
from datetime import datetime
from itertools import product

import joblib
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

from sklearn.metrics import (
    confusion_matrix,
    roc_auc_score
)

from xgboost import XGBClassifier

from scipy.stats import ks_2samp

import warnings

warnings.filterwarnings("ignore")


# =========================================================================
# OUTPUT DIRECTORY
# =========================================================================

OUTPUT_DIR = r"D:\Thesis\8-PKL-file"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


# =========================================================================
# OUTPUT FILE PATHS
# =========================================================================

HYPERPARAMETER_RESULTS_PATH = os.path.join(
    OUTPUT_DIR,
    "hyperparameter_search_results.csv"
)

MONITORING_LOG_PATH = os.path.join(
    OUTPUT_DIR,
    "monitoring_history.jsonl"
)

PKL_PATH = os.path.join(
    OUTPUT_DIR,
    "credit_risk_package.pkl"
)

METADATA_PATH = os.path.join(
    OUTPUT_DIR,
    "credit_risk_metadata.pkl"
)


# =========================================================================
# 2. LOAD ORIGINAL DATASET
# =========================================================================

df = pd.read_csv(
    r"D:\Thesis\4-Dataset\Loan_approval_data_2025.csv",
    sep=",",
    encoding="utf-8"
)


# =========================================================================
# 3. CHECK CORRUPTED ROWS
# =========================================================================

corrupted_rows = []

df["loan_status"] = df["loan_status"].astype(int)


for idx, row in df.iterrows():

    issues = []

    # ---------------------------------------------------------
    # Age validation
    # ---------------------------------------------------------

    if pd.notna(row["age"]) and (
        row["age"] < 18
        or row["age"] > 100
    ):

        issues.append(
            "Invalid age"
        )


    # ---------------------------------------------------------
    # Credit score validation
    # ---------------------------------------------------------

    if pd.notna(row["credit_score"]) and (
        row["credit_score"] < 300
        or row["credit_score"] > 850
    ):

        issues.append(
            "Invalid credit_score"
        )


    # ---------------------------------------------------------
    # Years employed
    # ---------------------------------------------------------

    if pd.notna(row["years_employed"]) and (
        row["years_employed"] < 0
        or row["years_employed"] > 60
    ):

        issues.append(
            "Invalid years_employed"
        )


    # ---------------------------------------------------------
    # Credit history
    # ---------------------------------------------------------

    if pd.notna(row["credit_history_years"]) and (
        row["credit_history_years"] < 0
        or row["credit_history_years"] > 80
    ):

        issues.append(
            "Invalid credit_history_years"
        )


    # ---------------------------------------------------------
    # Negative values
    # ---------------------------------------------------------

    if pd.notna(row["annual_income"]) and (
        row["annual_income"] < 0
    ):

        issues.append(
            "Negative annual_income"
        )


    if pd.notna(row["savings_assets"]) and (
        row["savings_assets"] < 0
    ):

        issues.append(
            "Negative savings_assets"
        )


    if pd.notna(row["current_debt"]) and (
        row["current_debt"] < 0
    ):

        issues.append(
            "Negative current_debt"
        )


    if pd.notna(row["loan_amount"]) and (
        row["loan_amount"] < 0
    ):

        issues.append(
            "Negative loan_amount"
        )


    # ---------------------------------------------------------
    # Interest rate
    # ---------------------------------------------------------

    if pd.notna(row["interest_rate"]) and (
        row["interest_rate"] < 0
        or row["interest_rate"] > 100
    ):

        issues.append(
            "Invalid interest_rate"
        )


    # ---------------------------------------------------------
    # Ratio validations
    # ---------------------------------------------------------

    if pd.notna(row["debt_to_income_ratio"]) and (
        row["debt_to_income_ratio"] < 0
        or row["debt_to_income_ratio"] > 100
    ):

        issues.append(
            "Invalid debt_to_income_ratio"
        )


    if pd.notna(row["loan_to_income_ratio"]) and (
        row["loan_to_income_ratio"] < 0
        or row["loan_to_income_ratio"] > 100
    ):

        issues.append(
            "Invalid loan_to_income_ratio"
        )


    if pd.notna(row["payment_to_income_ratio"]) and (
        row["payment_to_income_ratio"] < 0
        or row["payment_to_income_ratio"] > 100
    ):

        issues.append(
            "Invalid payment_to_income_ratio"
        )


    # ---------------------------------------------------------
    # Defaults / delinquencies / derogatory
    # ---------------------------------------------------------

    if pd.notna(row["defaults_on_file"]) and (
        row["defaults_on_file"] < 0
    ):

        issues.append(
            "Negative defaults_on_file"
        )


    if pd.notna(row["delinquencies_last_2yrs"]) and (
        row["delinquencies_last_2yrs"] < 0
    ):

        issues.append(
            "Negative delinquencies_last_2yrs"
        )


    if pd.notna(row["derogatory_marks"]) and (
        row["derogatory_marks"] < 0
    ):

        issues.append(
            "Negative derogatory_marks"
        )


    # ---------------------------------------------------------
    # Regex validation
    # ---------------------------------------------------------

    if pd.notna(row["occupation_status"]):

        if not re.match(
            r"^[A-Za-z\s\-]+$",
            str(row["occupation_status"])
        ):

            issues.append(
                "Malformed occupation_status"
            )


    if pd.notna(row["product_type"]):

        if not re.match(
            r"^[A-Za-z\s\-]+$",
            str(row["product_type"])
        ):

            issues.append(
                "Malformed product_type"
            )


    if pd.notna(row["loan_status"]):

        if int(row["loan_status"]) not in [0, 1]:

            issues.append(
                "Malformed loan_status"
            )


    # ---------------------------------------------------------
    # Logical inconsistencies
    # ---------------------------------------------------------

    if (
        pd.notna(row["credit_history_years"])
        and pd.notna(row["age"])
    ):

        if (
            row["credit_history_years"]
            > row["age"] - 18
        ):

            issues.append(
                "Credit history exceeds possible years"
            )


    if (
        pd.notna(row["years_employed"])
        and pd.notna(row["age"])
    ):

        if (
            row["years_employed"]
            > row["age"] - 18
        ):

            issues.append(
                "Years employed exceeds possible years"
            )


    # ---------------------------------------------------------
    # Missing critical values
    # ---------------------------------------------------------

    if (
        pd.isna(row["customer_id"])
        or pd.isna(row["loan_status"])
    ):

        issues.append(
            "Missing critical field"
        )


    # ---------------------------------------------------------
    # Store corrupted rows
    # ---------------------------------------------------------

    if issues:

        corrupted_rows.append(
            {
                "index": idx,

                "customer_id":
                    row["customer_id"],

                "issues":
                    ", ".join(issues),

                "row_data":
                    row.to_dict()
            }
        )


# =========================================================================
# 4. REMOVE INVALID ANNUAL INCOME VALUES
# =========================================================================

df = df[
    df["annual_income"].notna()
].copy()


df = df[
    df["annual_income"] > 0
].copy()


# =========================================================================
# 5. RECALCULATE RATIOS
# =========================================================================

df["loan_to_income_ratio"] = (
    df["loan_amount"]
    /
    df["annual_income"]
)


df["debt_to_income_ratio"] = (
    df["current_debt"]
    /
    df["annual_income"]
)


# =========================================================================
# 6. REMOVE FIRST ROW
# =========================================================================

df_corrected = (
    df.iloc[1:]
    .copy()
)


# =========================================================================
# 7. SAVE CORRECTED DATASET
# =========================================================================

corrected_dataset_path = (
    r"D:\Thesis\4-Dataset\Loan_approval_data_2025-1.csv"
)


df_corrected.to_csv(
    corrected_dataset_path,
    index=False
)


# =========================================================================
# 8. RELOAD CORRECTED DATASET
# =========================================================================

df = pd.read_csv(
    corrected_dataset_path
)


# =========================================================================
# 9. FIX COLUMN DATA TYPES
# =========================================================================

df["age"] = (
    df["age"]
    .astype(int)
)


df["defaults_on_file"] = (
    df["defaults_on_file"]
    .astype(int)
)


df["delinquencies_last_2yrs"] = (
    df["delinquencies_last_2yrs"]
    .astype(int)
)


df["derogatory_marks"] = (
    df["derogatory_marks"]
    .astype(int)
)


df["loan_status"] = (
    df["loan_status"]
    .astype(int)
)


# =========================================================================
# 10. HANDLE MISSING VALUES
# =========================================================================

df.dropna(
    inplace=True
)


# =========================================================================
# 11. REMOVE DUPLICATES
# =========================================================================

df = (
    df
    .drop_duplicates()
    .reset_index(drop=True)
)


# =========================================================================
# 12. CLASS IMBALANCE
# =========================================================================

class_counts = (
    df["loan_status"]
    .value_counts()
)


class_percent = (
    df["loan_status"]
    .value_counts(
        normalize=True
    )
    * 100
)


majority_class_count = (
    class_counts.max()
)


minority_class_count = (
    class_counts.min()
)


imbalance_ratio = (
    majority_class_count
    /
    minority_class_count
)


# =========================================================================
# 13. FEATURE GROUPS
# =========================================================================

log_features = [
    "current_debt",
    "annual_income",
    "savings_assets"
]


scale_features = [
    "loan_amount",
    "credit_history_years",
    "debt_to_income_ratio",
    "loan_to_income_ratio",
    "age",
    "payment_to_income_ratio",
    "credit_score",
    "interest_rate"
]


binary_features = [
    "delinquencies_last_2yrs",
    "derogatory_marks"
]


categorical_features = [
    "occupation_status",
    "product_type",
    "loan_intent"
]


# =========================================================================
# 14. TRAIN / VALIDATION / TEST SPLIT
# =========================================================================

X = df.drop(
    columns=[
        "loan_status",
        "customer_id"
    ]
)


y = df[
    "loan_status"
]


X_train, X_temp, y_train, y_temp = (
    train_test_split(
        X,
        y,
        test_size=0.2,
        stratify=y,
        random_state=42
    )
)


X_val, X_test, y_val, y_test = (
    train_test_split(
        X_temp,
        y_temp,
        test_size=0.5,
        stratify=y_temp,
        random_state=42
    )
)


X_train = X_train.copy()
X_val = X_val.copy()
X_test = X_test.copy()


# =========================================================================
# 15. LOG TRANSFORM
# =========================================================================

for col in log_features:

    X_train[
        f"{col}_log"
    ] = np.log1p(
        X_train[col]
    )


    X_val[
        f"{col}_log"
    ] = np.log1p(
        X_val[col]
    )


    X_test[
        f"{col}_log"
    ] = np.log1p(
        X_test[col]
    )


# =========================================================================
# 16. SCALE ORIGINAL NUMERICAL FEATURES
# =========================================================================

scalers = {}


for col in scale_features:

    scaler = StandardScaler()


    X_train[
        f"{col}_scaled"
    ] = scaler.fit_transform(
        X_train[[col]]
    )


    scalers[col] = scaler


    X_val[
        f"{col}_scaled"
    ] = scaler.transform(
        X_val[[col]]
    )


    X_test[
        f"{col}_scaled"
    ] = scaler.transform(
        X_test[[col]]
    )


# =========================================================================
# 17. SCALE LOG FEATURES
# =========================================================================

log_scaled_features = [
    f"{col}_log"
    for col in log_features
]


for col in log_scaled_features:

    scaler = StandardScaler()


    X_train[
        f"{col}_scaled"
    ] = scaler.fit_transform(
        X_train[[col]]
    )


    scalers[col] = scaler


    X_val[
        f"{col}_scaled"
    ] = scaler.transform(
        X_val[[col]]
    )


    X_test[
        f"{col}_scaled"
    ] = scaler.transform(
        X_test[[col]]
    )


# =========================================================================
# 18. BINARY FEATURES
# =========================================================================

for col in binary_features:

    X_train[
        f"{col}_binary"
    ] = (
        X_train[col] > 0
    ).astype(int)


    X_val[
        f"{col}_binary"
    ] = (
        X_val[col] > 0
    ).astype(int)


    X_test[
        f"{col}_binary"
    ] = (
        X_test[col] > 0
    ).astype(int)


# =========================================================================
# 19. ONE-HOT ENCODING
# =========================================================================

X_train_encoded = pd.get_dummies(
    X_train,
    columns=categorical_features,
    drop_first=True
)


X_val_encoded = pd.get_dummies(
    X_val,
    columns=categorical_features,
    drop_first=True
)


X_test_encoded = pd.get_dummies(
    X_test,
    columns=categorical_features,
    drop_first=True
)


X_val_encoded = (
    X_val_encoded
    .reindex(
        columns=X_train_encoded.columns,
        fill_value=0
    )
)


X_test_encoded = (
    X_test_encoded
    .reindex(
        columns=X_train_encoded.columns,
        fill_value=0
    )
)


# =========================================================================
# 20. DROP ORIGINAL TRANSFORMED FEATURES
# =========================================================================

features_to_drop = (
    log_features
    +
    scale_features
    +
    binary_features
)


X_train_encoded = (
    X_train_encoded
    .drop(
        columns=features_to_drop,
        errors="ignore"
    )
)


X_val_encoded = (
    X_val_encoded
    .drop(
        columns=features_to_drop,
        errors="ignore"
    )
)


X_test_encoded = (
    X_test_encoded
    .drop(
        columns=features_to_drop,
        errors="ignore"
    )
)


# =========================================================================
# 21. FINAL FEATURE SELECTION
# =========================================================================

final_features = [

    "credit_score_scaled",

    "debt_to_income_ratio_scaled",

    "age_scaled",

    "credit_history_years_scaled",

    "defaults_on_file",

    "annual_income_log_scaled",

    "current_debt_log_scaled",

    "loan_amount_scaled",

    "delinquencies_last_2yrs_binary",

    "derogatory_marks_binary",

    "interest_rate_scaled",

    "payment_to_income_ratio_scaled",

    "loan_to_income_ratio_scaled",

    "savings_assets_log_scaled"

]


categorical_cols = [

    col

    for col
    in X_train_encoded.columns

    if any(
        col.startswith(
            cat + "_"
        )
        for cat
        in categorical_features
    )

]


final_features += (
    categorical_cols
)


final_features = [

    f

    for f
    in final_features

    if f
    in X_train_encoded.columns

]


X_train_final = (
    X_train_encoded[
        final_features
    ]
)


X_val_final = (
    X_val_encoded[
        final_features
    ]
)


X_test_final = (
    X_test_encoded[
        final_features
    ]
)


# =========================================================================
# 22. CLASS WEIGHTS
# =========================================================================

class_weights = (
    compute_class_weight(
        class_weight="balanced",
        classes=np.unique(
            y_train
        ),
        y=y_train
    )
)


sample_weights = np.array(
    [
        class_weights[i]
        for i in y_train
    ]
)


# =========================================================================
# 23. HYPERPARAMETER GRID
# =========================================================================

param_grid = {

    "n_estimators": [
        200,
        300,
        400,
        500
    ],

    "learning_rate": [
        0.03,
        0.05,
        0.07,
        0.1
    ],

    "max_depth": [
        3,
        4,
        5,
        6
    ],

    "min_child_weight": [
        1,
        3,
        5,
        7
    ]

}


fixed_params = {

    "subsample": 0.8,

    "colsample_bytree": 0.8,

    "objective":
        "binary:logistic",

    "eval_metric":
        "logloss",

    "random_state":
        42

}


param_combinations = list(
    product(

        param_grid[
            "n_estimators"
        ],

        param_grid[
            "learning_rate"
        ],

        param_grid[
            "max_depth"
        ],

        param_grid[
            "min_child_weight"
        ]

    )
)


# =========================================================================
# 24. GRID SEARCH
# =========================================================================

results = []


for (
    n_est,
    lr,
    max_d,
    min_cw
) in param_combinations:

    model = XGBClassifier(

        n_estimators=n_est,

        learning_rate=lr,

        max_depth=max_d,

        min_child_weight=min_cw,

        early_stopping_rounds=30,

        **fixed_params

    )


    model.fit(

        X_train_final,

        y_train,

        sample_weight=sample_weights,

        eval_set=[
            (
                X_val_final,
                y_val
            )
        ],

        verbose=0

    )


    y_val_pred_proba = (
        model
        .predict_proba(
            X_val_final
        )[:, 1]
    )


    val_auc = (
        roc_auc_score(
            y_val,
            y_val_pred_proba
        )
    )


    thresholds = np.arange(
        0.15,
        0.6,
        0.01
    )


    best_score = np.inf

    best_thresh = 0.5

    best_fn = 0

    best_fp = 0


    for thresh in thresholds:

        y_val_pred = (
            y_val_pred_proba
            >= thresh
        ).astype(int)


        tn, fp, fn, tp = (
            confusion_matrix(
                y_val,
                y_val_pred,
                labels=[0, 1]
            ).ravel()
        )


        cost = (
            2.5 * fn
            +
            1.0 * fp
        )


        if fn > 0:

            fp_fn_ratio = (
                fp
                /
                fn
            )

        else:

            fp_fn_ratio = fp


        if fp_fn_ratio < 3:

            balance_penalty = 100

        elif fp_fn_ratio > 20:

            balance_penalty = 50

        else:

            balance_penalty = 0


        total_score = (
            cost
            +
            balance_penalty
        )


        if total_score < best_score:

            best_score = (
                total_score
            )

            best_thresh = (
                thresh
            )

            best_fn = (
                fn
            )

            best_fp = (
                fp
            )


    results.append(
        {

            "n_estimators":
                n_est,

            "learning_rate":
                lr,

            "max_depth":
                max_d,

            "min_child_weight":
                min_cw,

            "val_auc":
                val_auc,

            "best_threshold":
                best_thresh,

            "fn":
                best_fn,

            "fp":
                best_fp,

            "fp_fn_ratio":
                (
                    best_fp / best_fn
                    if best_fn > 0
                    else best_fp
                ),

            "total_errors":
                best_fn + best_fp,

            "balance_score":
                best_score

        }
    )


# =========================================================================
# 25. GRID SEARCH RESULTS
# =========================================================================

results_df = pd.DataFrame(
    results
)


results_df[
    "auc_rank"
] = (
    results_df[
        "val_auc"
    ]
    .rank(
        ascending=False
    )
)


results_df[
    "error_rank"
] = (
    results_df[
        "total_errors"
    ]
    .rank(
        ascending=True
    )
)


results_df[
    "balance_rank"
] = (
    results_df[
        "balance_score"
    ]
    .rank(
        ascending=True
    )
)


results_df[
    "composite_score"
] = (

    0.4
    *
    results_df[
        "auc_rank"
    ]

    +

    0.3
    *
    results_df[
        "error_rank"
    ]

    +

    0.3
    *
    results_df[
        "balance_rank"
    ]

)


results_df = (
    results_df
    .sort_values(
        "composite_score"
    )
)


# =========================================================================
# 26. BEST PARAMETERS
# =========================================================================

best_params = (
    results_df.iloc[0]
)


# =========================================================================
# 27. TRAIN FINAL MODEL
# =========================================================================

final_model = XGBClassifier(

    n_estimators=int(
        best_params[
            "n_estimators"
        ]
    ),

    learning_rate=float(
        best_params[
            "learning_rate"
        ]
    ),

    max_depth=int(
        best_params[
            "max_depth"
        ]
    ),

    min_child_weight=int(
        best_params[
            "min_child_weight"
        ]
    ),

    early_stopping_rounds=30,

    **fixed_params

)


final_model.fit(

    X_train_final,

    y_train,

    sample_weight=sample_weights,

    eval_set=[

        (
            X_train_final,
            y_train
        ),

        (
            X_val_final,
            y_val
        )

    ],

    verbose=50

)


# =========================================================================
# 28. FINAL PREDICTIONS
# =========================================================================

y_train_pred_proba = (
    final_model
    .predict_proba(
        X_train_final
    )[:, 1]
)


y_val_pred_proba = (
    final_model
    .predict_proba(
        X_val_final
    )[:, 1]
)


y_test_pred_proba = (
    final_model
    .predict_proba(
        X_test_final
    )[:, 1]
)


# =========================================================================
# 29. AUC
# =========================================================================

train_auc = (
    roc_auc_score(
        y_train,
        y_train_pred_proba
    )
)


val_auc = (
    roc_auc_score(
        y_val,
        y_val_pred_proba
    )
)


test_auc = (
    roc_auc_score(
        y_test,
        y_test_pred_proba
    )
)


# =========================================================================
# 30. SINGLE COST-OPTIMIZED THRESHOLD
# =========================================================================

best_threshold = float(
    best_params[
        "best_threshold"
    ]
)


y_test_pred = (
    y_test_pred_proba
    >= best_threshold
).astype(int)


# =========================================================================
# 31. TEST CONFUSION MATRIX
# =========================================================================

cm = confusion_matrix(
    y_test,
    y_test_pred,
    labels=[0, 1]
)


tn, fp, fn, tp = (
    cm.ravel()
)


# =========================================================================
# 32. TEST METRICS
# =========================================================================

test_accuracy = (
    (
        y_test_pred
        ==
        y_test
    ).mean()
)


test_precision = (
    tp / (tp + fp)
    if (tp + fp) > 0
    else 0
)


test_recall = (
    tp / (tp + fn)
    if (tp + fn) > 0
    else 0
)


test_f1 = (

    2
    *
    test_precision
    *
    test_recall

    /

    (
        test_precision
        +
        test_recall
    )

    if (
        test_precision
        +
        test_recall
    ) > 0

    else 0

)


test_cost = (
    2.5 * fn
    +
    1.0 * fp
)


fp_fn_ratio = (
    fp / fn
    if fn > 0
    else fp
)


# =========================================================================
# 33. RELIABILITY MONITORING REFERENCE
# =========================================================================

X_reference = (
    X_val_final.copy()
)


y_reference = (
    y_val.copy()
)


y_reference_pred_proba = (
    final_model
    .predict_proba(
        X_reference
    )[:, 1]
)


# =========================================================================
# 34. REFERENCE STATISTICS
# =========================================================================

reference_stats = {}


for feature in final_features:

    feature_data = (
        X_reference[
            feature
        ]
    )


    is_numeric = (
        str(feature_data.dtype)
        in [
            "float64",
            "int64",
            "float32",
            "int32"
        ]
    )


    is_binary = (
        feature_data.nunique()
        <= 2
    )


    if (
        is_numeric
        and not is_binary
    ):

        reference_stats[
            feature
        ] = {

            "mean":
                float(
                    feature_data.mean()
                ),

            "std":
                float(
                    feature_data.std()
                ),

            "min":
                float(
                    feature_data.min()
                ),

            "max":
                float(
                    feature_data.max()
                ),

            "q25":
                float(
                    feature_data.quantile(
                        0.25
                    )
                ),

            "q50":
                float(
                    feature_data.quantile(
                        0.50
                    )
                ),

            "q75":
                float(
                    feature_data.quantile(
                        0.75
                    )
                ),

            "q90":
                float(
                    feature_data.quantile(
                        0.90
                    )
                ),

            "q95":
                float(
                    feature_data.quantile(
                        0.95
                    )
                )

        }


    else:

        mode_values = (
            feature_data.mode()
        )


        reference_stats[
            feature
        ] = {

            "mean":
                (
                    float(
                        feature_data.mean()
                    )
                    if is_numeric
                    else None
                ),

            "std":
                (
                    float(
                        feature_data.std()
                    )
                    if is_numeric
                    else None
                ),

            "min":
                (
                    float(
                        feature_data.min()
                    )
                    if is_numeric
                    else None
                ),

            "max":
                (
                    float(
                        feature_data.max()
                    )
                    if is_numeric
                    else None
                ),

            "unique_values":
                int(
                    feature_data.nunique()
                ),

            "mode":
                (
                    mode_values.iloc[0]
                    if len(mode_values) > 0
                    else None
                )

        }


# =========================================================================
# 35. REFERENCE PREDICTION STATISTICS
# =========================================================================

reference_pred_stats = {

    "mean_prob":
        float(
            y_reference_pred_proba.mean()
        ),

    "std_prob":
        float(
            y_reference_pred_proba.std()
        ),

    "median_prob":
        float(
            np.median(
                y_reference_pred_proba
            )
        ),

    "approval_rate":
        float(
            (
                y_reference_pred_proba
                <
                best_threshold
            ).mean()
        ),

    "rejection_rate":
        float(
            (
                y_reference_pred_proba
                >=
                best_threshold
            ).mean()
        )

}


# =========================================================================
# 36. REFERENCE PERFORMANCE
# =========================================================================

y_reference_pred = (
    y_reference_pred_proba
    >= best_threshold
).astype(int)


cm_ref = confusion_matrix(
    y_reference,
    y_reference_pred,
    labels=[0, 1]
)


tn_ref, fp_ref, fn_ref, tp_ref = (
    cm_ref.ravel()
)


reference_performance = {

    "auc":
        float(
            roc_auc_score(
                y_reference,
                y_reference_pred_proba
            )
        ),

    "tn":
        int(tn_ref),

    "fp":
        int(fp_ref),

    "fn":
        int(fn_ref),

    "tp":
        int(tp_ref),

    "fp_fn_ratio":
        float(
            fp_ref / fn_ref
            if fn_ref > 0
            else fp_ref
        ),

    "total_errors":
        int(
            fp_ref + fn_ref
        ),

    "cost":
        float(
            2.5 * fn_ref
            + 1.0 * fp_ref
        ),

    "threshold":
        float(
            best_threshold
        )

}


# =========================================================================
# 37. PSI FUNCTION
# =========================================================================

def calculate_psi(
    reference_data,
    new_data,
    bins=10
):

    if reference_data.nunique() <= 10:

        ref_counts = (
            reference_data
            .value_counts(
                normalize=True,
                dropna=False
            )
        )


        new_counts = (
            new_data
            .value_counts(
                normalize=True,
                dropna=False
            )
        )


        all_categories = (
            set(ref_counts.index)
            |
            set(new_counts.index)
        )


        ref_pct = pd.Series(
            {
                cat:
                ref_counts.get(
                    cat,
                    0.0001
                )

                for cat
                in all_categories
            }
        )


        new_pct = pd.Series(
            {
                cat:
                new_counts.get(
                    cat,
                    0.0001
                )

                for cat
                in all_categories
            }
        )


    else:

        breakpoints = (
            np.percentile(
                reference_data,
                np.linspace(
                    0,
                    100,
                    bins + 1
                )
            )
        )


        breakpoints = np.unique(
            breakpoints
        )


        if len(breakpoints) < 2:

            return 0.0


        ref_binned = pd.cut(
            reference_data,
            bins=breakpoints,
            include_lowest=True,
            duplicates="drop"
        )


        new_binned = pd.cut(
            new_data,
            bins=breakpoints,
            include_lowest=True,
            duplicates="drop"
        )


        ref_pct = (
            ref_binned
            .value_counts(
                normalize=True,
                dropna=False
            )
        )


        new_pct = (
            new_binned
            .value_counts(
                normalize=True,
                dropna=False
            )
        )


        all_bins = (
            set(ref_pct.index)
            |
            set(new_pct.index)
        )


        ref_pct = pd.Series(
            {
                b:
                ref_pct.get(
                    b,
                    0.0001
                )

                for b
                in all_bins
            }
        )


        new_pct = pd.Series(
            {
                b:
                new_pct.get(
                    b,
                    0.0001
                )

                for b
                in all_bins
            }
        )


    ref_pct = (
        ref_pct
        .replace(
            0,
            0.0001
        )
    )


    new_pct = (
        new_pct
        .replace(
            0,
            0.0001
        )
    )


    psi = np.sum(
        (
            new_pct
            -
            ref_pct
        )
        *
        np.log(
            new_pct
            /
            ref_pct
        )
    )


    return float(
        psi
    )


# =========================================================================
# 38. KS STATISTIC
# =========================================================================

def calculate_ks_statistic(
    reference_data,
    new_data
):

    ref_clean = (
        reference_data
        .dropna()
    )


    new_clean = (
        new_data
        .dropna()
    )


    if (
        len(ref_clean) == 0
        or len(new_clean) == 0
    ):

        return 0.0, 1.0


    ks_stat, p_value = (
        ks_2samp(
            ref_clean,
            new_clean
        )
    )


    return (
        float(ks_stat),
        float(p_value)
    )


# =========================================================================
# 39. DATA DRIFT MONITORING
# =========================================================================

def monitor_data_drift(
    X_new,
    X_reference,
    feature_names,
    psi_threshold_moderate=0.1,
    psi_threshold_severe=0.2,
    ks_threshold=0.1
):

    drift_results = []


    for feature in feature_names:

        ref_data = (
            X_reference[
                feature
            ]
        )


        new_data = (
            X_new[
                feature
            ]
        )


        psi = calculate_psi(
            ref_data,
            new_data
        )


        ks_stat, ks_pvalue = (
            calculate_ks_statistic(
                ref_data,
                new_data
            )
        )


        if (
            psi >= psi_threshold_severe
            or ks_stat >= ks_threshold
        ):

            status = "SEVERE"


        elif (
            psi >= psi_threshold_moderate
        ):

            status = "MODERATE"


        else:

            status = "STABLE"


        is_numeric = (
            pd.api.types
            .is_numeric_dtype(
                ref_data
            )
        )


        drift_results.append(

            {

                "feature":
                    feature,

                "psi":
                    psi,

                "ks_statistic":
                    ks_stat,

                "ks_pvalue":
                    ks_pvalue,

                "status":
                    status,

                "mean_ref":
                    (
                        float(
                            ref_data.mean()
                        )
                        if is_numeric
                        else None
                    ),

                "mean_new":
                    (
                        float(
                            new_data.mean()
                        )
                        if is_numeric
                        else None
                    )

            }

        )


    drift_df = (
        pd.DataFrame(
            drift_results
        )
        .sort_values(
            "psi",
            ascending=False
        )
    )


    return drift_df


# =========================================================================
# 40. PREDICTION DRIFT MONITORING
# =========================================================================

def monitor_prediction_drift(
    y_new_proba,
    y_reference_proba,
    threshold,
    prob_diff_threshold=0.05,
    approval_rate_diff_threshold=0.10
):

    new_stats = {

        "mean_prob":
            float(
                y_new_proba.mean()
            ),

        "std_prob":
            float(
                y_new_proba.std()
            ),

        "median_prob":
            float(
                np.median(
                    y_new_proba
                )
            ),

        "approval_rate":
            float(
                (
                    y_new_proba
                    <
                    threshold
                ).mean()
            ),

        "rejection_rate":
            float(
                (
                    y_new_proba
                    >=
                    threshold
                ).mean()
            )

    }


    ref_stats = {

        "mean_prob":
            float(
                y_reference_proba.mean()
            ),

        "std_prob":
            float(
                y_reference_proba.std()
            ),

        "median_prob":
            float(
                np.median(
                    y_reference_proba
                )
            ),

        "approval_rate":
            float(
                (
                    y_reference_proba
                    <
                    threshold
                ).mean()
            ),

        "rejection_rate":
            float(
                (
                    y_reference_proba
                    >=
                    threshold
                ).mean()
            )

    }


    prob_diff = abs(

        new_stats[
            "mean_prob"
        ]

        -

        ref_stats[
            "mean_prob"
        ]

    )


    approval_rate_diff = abs(

        new_stats[
            "approval_rate"
        ]

        -

        ref_stats[
            "approval_rate"
        ]

    )


    alerts = []


    if (
        prob_diff
        >
        prob_diff_threshold
    ):

        alerts.append(
            "Mean probability shifted."
        )


    if (
        approval_rate_diff
        >
        approval_rate_diff_threshold
    ):

        alerts.append(
            "Approval rate changed."
        )


    pred_psi = calculate_psi(

        pd.Series(
            y_reference_proba
        ),

        pd.Series(
            y_new_proba
        ),

        bins=10

    )


    if pred_psi > 0.2:

        alerts.append(
            "Prediction PSI is severe."
        )


    elif pred_psi > 0.1:

        alerts.append(
            "Prediction PSI is moderate."
        )


    return {

        "new_stats":
            new_stats,

        "ref_stats":
            ref_stats,

        "prob_diff":
            float(
                prob_diff
            ),

        "approval_rate_diff":
            float(
                approval_rate_diff
            ),

        "pred_psi":
            float(
                pred_psi
            ),

        "alerts":
            alerts,

        "drift_detected":
            len(alerts) > 0

    }


# =========================================================================
# 41. PERFORMANCE MONITORING
# =========================================================================

def monitor_performance(
    y_true,
    y_pred_proba,
    threshold,
    reference_performance,
    auc_threshold=0.95,
    fn_multiplier=1.5
):

    y_pred = (
        y_pred_proba
        >= threshold
    ).astype(int)


    auc = (
        roc_auc_score(
            y_true,
            y_pred_proba
        )
    )


    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    )


    tn, fp, fn, tp = (
        cm.ravel()
    )


    fp_fn_ratio = (
        fp / fn
        if fn > 0
        else fp
    )


    total_errors = (
        fp + fn
    )


    cost = (
        2.5 * fn
        +
        1.0 * fp
    )


    alerts = []


    if auc < auc_threshold:

        alerts.append(
            "AUC below monitoring threshold."
        )


    if (
        auc
        <
        reference_performance[
            "auc"
        ]
        -
        0.02
    ):

        alerts.append(
            "AUC decreased from reference."
        )


    if (
        fn
        >
        reference_performance[
            "fn"
        ]
        *
        fn_multiplier
    ):

        alerts.append(
            "False negatives increased."
        )


    if fp_fn_ratio < 3:

        alerts.append(
            "FP/FN ratio is too low."
        )


    if (
        total_errors
        >
        reference_performance[
            "total_errors"
        ]
        *
        1.3
    ):

        alerts.append(
            "Total errors increased."
        )


    return {

        "auc":
            float(auc),

        "tn":
            int(tn),

        "fp":
            int(fp),

        "fn":
            int(fn),

        "tp":
            int(tp),

        "fp_fn_ratio":
            float(fp_fn_ratio),

        "total_errors":
            int(total_errors),

        "cost":
            float(cost),

        "alerts":
            alerts,

        "performance_degraded":
            len(alerts) > 0

    }


# =========================================================================
# 42. SELECTIVE PREDICTION
# =========================================================================

def selective_prediction(
    y_proba,
    t1,
    t2
):

    decisions = np.zeros(
        len(y_proba),
        dtype=int
    )


    decisions[
        y_proba < t1
    ] = 0


    decisions[
        y_proba > t2
    ] = 1


    decisions[
        (
            y_proba >= t1
        )
        &
        (
            y_proba <= t2
        )
    ] = 2


    return decisions


# =========================================================================
# 43. OPTIMIZE SELECTIVE THRESHOLDS
# =========================================================================

def optimize_selective_thresholds(
    y_true,
    y_proba,
    max_manual_review_rate=0.20,
    fn_cost=2.5,
    fp_cost=1.0
):

    best_cost = np.inf

    best_t1 = 0.3

    best_t2 = 0.5

    best_metrics = None


    t1_range = np.arange(
        0.20,
        0.45,
        0.05
    )


    t2_range = np.arange(
        0.45,
        0.70,
        0.05
    )


    results = []


    for t1 in t1_range:

        for t2 in t2_range:

            if t1 >= t2:

                continue


            decisions = (
                selective_prediction(
                    y_proba,
                    t1,
                    t2
                )
            )


            auto_mask = (
                decisions != 2
            )


            manual_mask = (
                decisions == 2
            )


            if auto_mask.sum() == 0:

                continue


            y_true_auto = (
                y_true[
                    auto_mask
                ]
            )


            y_pred_auto = (
                decisions[
                    auto_mask
                ]
            )


            cm = confusion_matrix(
                y_true_auto,
                y_pred_auto,
                labels=[0, 1]
            )


            if cm.shape != (2, 2):

                continue


            tn, fp, fn, tp = (
                cm.ravel()
            )


            manual_review_rate = (
                manual_mask.sum()
                /
                len(y_proba)
            )


            if (
                manual_review_rate
                >
                max_manual_review_rate
            ):

                continue


            cost = (
                fn_cost * fn
                +
                fp_cost * fp
            )


            fp_fn_ratio = (
                fp / fn
                if fn > 0
                else fp
            )


            if fp_fn_ratio < 3:

                cost += 100

            elif fp_fn_ratio > 20:

                cost += 50


            results.append(

                {

                    "t1":
                        float(t1),

                    "t2":
                        float(t2),

                    "cost":
                        float(cost),

                    "fn":
                        int(fn),

                    "fp":
                        int(fp),

                    "fp_fn_ratio":
                        float(
                            fp_fn_ratio
                        ),

                    "manual_review_rate":
                        float(
                            manual_review_rate
                        ),

                    "auto_decisions":
                        int(
                            auto_mask.sum()
                        ),

                    "tn":
                        int(tn),

                    "tp":
                        int(tp)

                }

            )


            if cost < best_cost:

                best_cost = cost

                best_t1 = t1

                best_t2 = t2

                best_metrics = {

                    "cost":
                        float(
                            cost
                        ),

                    "fn":
                        int(
                            fn
                        ),

                    "fp":
                        int(
                            fp
                        ),

                    "tn":
                        int(
                            tn
                        ),

                    "tp":
                        int(
                            tp
                        ),

                    "fp_fn_ratio":
                        float(
                            fp_fn_ratio
                        ),

                    "manual_review_rate":
                        float(
                            manual_review_rate
                        ),

                    "auto_decisions":
                        int(
                            auto_mask.sum()
                        )

                }


    results_df = (
        pd.DataFrame(
            results
        )
    )


    if not results_df.empty:

        results_df = (
            results_df
            .sort_values(
                "cost"
            )
        )


    return (
        float(best_t1),
        float(best_t2),
        best_metrics,
        results_df
    )


# =========================================================================
# 44. OPTIMIZE SELECTIVE THRESHOLDS ON VALIDATION SET
# =========================================================================

best_t1, best_t2, selective_metrics, selective_results = (

    optimize_selective_thresholds(

        y_val.values,

        y_val_pred_proba,

        max_manual_review_rate=0.20,

        fn_cost=2.5,

        fp_cost=1.0

    )

)


# =========================================================================
# 45. TEST SELECTIVE PREDICTION
# =========================================================================

test_decisions = (
    selective_prediction(
        y_test_pred_proba,
        best_t1,
        best_t2
    )
)


auto_accept_mask = (
    test_decisions == 0
)


auto_reject_mask = (
    test_decisions == 1
)


manual_review_mask = (
    test_decisions == 2
)


auto_mask = (
    test_decisions != 2
)


y_test_auto = (
    y_test[
        auto_mask
    ]
)


y_pred_auto = (
    test_decisions[
        auto_mask
    ]
)


if len(y_test_auto) > 0:

    cm_auto = (
        confusion_matrix(
            y_test_auto,
            y_pred_auto,
            labels=[0, 1]
        )
    )


    if cm_auto.shape == (2, 2):

        (
            tn_auto,
            fp_auto,
            fn_auto,
            tp_auto
        ) = cm_auto.ravel()


        selective_cost = (
            2.5 * fn_auto
            +
            1.0 * fp_auto
        )


        selective_fp_fn_ratio = (
            fp_auto / fn_auto
            if fn_auto > 0
            else fp_auto
        )


    else:

        tn_auto = 0

        fp_auto = 0

        fn_auto = 0

        tp_auto = 0

        selective_cost = 0

        selective_fp_fn_ratio = 0


else:

    tn_auto = 0

    fp_auto = 0

    fn_auto = 0

    tp_auto = 0

    selective_cost = 0

    selective_fp_fn_ratio = 0


# =========================================================================
# 46. MANUAL REVIEW ZONE
# =========================================================================

if manual_review_mask.sum() > 0:

    manual_review_actual = (
        y_test[
            manual_review_mask
        ]
    )


    manual_review_proba = (
        y_test_pred_proba[
            manual_review_mask
        ]
    )


    manual_review_actual_default_rate = float(
        manual_review_actual.mean()
    )


    manual_review_probability_mean = float(
        manual_review_proba.mean()
    )


    manual_review_probability_min = float(
        manual_review_proba.min()
    )


    manual_review_probability_max = float(
        manual_review_proba.max()
    )


else:

    manual_review_actual_default_rate = 0.0

    manual_review_probability_mean = 0.0

    manual_review_probability_min = 0.0

    manual_review_probability_max = 0.0


# =========================================================================
# 47. NIGHTLY MONITORING FUNCTIONS
# =========================================================================

def get_new_data_for_monitoring():

    return (
        X_test_final,
        y_test
    )


def send_alert(
    record,
    severe_drift,
    pred_drift_report,
    perf_report
):

    alert_data = {

        "timestamp":
            record[
                "timestamp"
            ],

        "severe_drift":
            severe_drift[
                "feature"
            ].tolist(),

        "prediction_alerts":
            pred_drift_report[
                "alerts"
            ],

        "performance_alerts":
            perf_report[
                "alerts"
            ]

    }


    return alert_data


def nightly_monitoring_job():

    timestamp = (
        datetime.now()
        .isoformat()
    )


    try:

        X_new, y_new = (
            get_new_data_for_monitoring()
        )


        y_new_pred_proba = (
            final_model
            .predict_proba(
                X_new
            )[:, 1]
        )


        drift_report = (
            monitor_data_drift(
                X_new,
                X_reference,
                final_features
            )
        )


        severe_drift = (
            drift_report[
                drift_report[
                    "status"
                ] == "SEVERE"
            ]
        )


        moderate_drift = (
            drift_report[
                drift_report[
                    "status"
                ] == "MODERATE"
            ]
        )


        pred_drift_report = (
            monitor_prediction_drift(
                y_new_pred_proba,
                y_reference_pred_proba,
                best_threshold
            )
        )


        perf_report = (
            monitor_performance(
                y_new,
                y_new_pred_proba,
                best_threshold,
                reference_performance
            )
        )


        action_required = (

            len(severe_drift) > 0

            or

            pred_drift_report[
                "drift_detected"
            ]

            or

            perf_report[
                "performance_degraded"
            ]

        )


        record = {

            "timestamp":
                timestamp,

            "n_samples":
                int(
                    len(X_new)
                ),

            "severe_drift_features":
                severe_drift[
                    "feature"
                ].tolist(),

            "moderate_drift_features":
                moderate_drift[
                    "feature"
                ].tolist(),

            "prediction_psi":
                float(
                    pred_drift_report[
                        "pred_psi"
                    ]
                ),

            "prediction_drift_detected":
                bool(
                    pred_drift_report[
                        "drift_detected"
                    ]
                ),

            "mean_predicted_prob":
                float(
                    y_new_pred_proba.mean()
                ),

            "approval_rate":
                float(
                    (
                        y_new_pred_proba
                        <
                        best_threshold
                    ).mean()
                ),

            "auc":
                float(
                    perf_report[
                        "auc"
                    ]
                ),

            "fn":
                int(
                    perf_report[
                        "fn"
                    ]
                ),

            "fp":
                int(
                    perf_report[
                        "fp"
                    ]
                ),

            "cost":
                float(
                    perf_report[
                        "cost"
                    ]
                ),

            "action_required":
                bool(
                    action_required
                )

        }


        with open(
            MONITORING_LOG_PATH,
            "a",
            encoding="utf-8"
        ) as f:

            f.write(
                json.dumps(
                    record
                )
                +
                "\n"
            )


        if action_required:

            alert_data = (
                send_alert(
                    record,
                    severe_drift,
                    pred_drift_report,
                    perf_report
                )
            )

        else:

            alert_data = None


        return {

            "record":
                record,

            "drift_report":
                drift_report,

            "prediction_drift":
                pred_drift_report,

            "performance":
                perf_report,

            "alert":
                alert_data

        }


    except Exception as e:

        error_record = {

            "timestamp":
                timestamp,

            "status":
                "JOB_FAILED",

            "error":
                str(e),

            "traceback":
                traceback.format_exc()

        }


        with open(
            MONITORING_LOG_PATH,
            "a",
            encoding="utf-8"
        ) as f:

            f.write(
                json.dumps(
                    error_record
                )
                +
                "\n"
            )


        return {

            "error":
                error_record

        }


# =========================================================================
# 48. RUN ONE MONITORING CHECK
# =========================================================================

monitoring_result = (
    nightly_monitoring_job()
)


# =========================================================================
# 49. CREATE COMPLETE PKL PACKAGE
# =========================================================================

credit_risk_package = {

    # ---------------------------------------------------------
    # MODEL
    # ---------------------------------------------------------

    "model":
        final_model,


    # ---------------------------------------------------------
    # PREPROCESSING
    # ---------------------------------------------------------

    "scalers":
        scalers,

    "log_features":
        log_features,

    "scale_features":
        scale_features,

    "binary_features":
        binary_features,

    "categorical_features":
        categorical_features,


    # ---------------------------------------------------------
    # FEATURE STRUCTURE
    # ---------------------------------------------------------

    "final_features":
        final_features,

    "training_encoded_columns":
        X_train_encoded.columns.tolist(),


    # ---------------------------------------------------------
    # CATEGORICAL VALUES
    # ---------------------------------------------------------

    "occupation_values":
        sorted(
            df[
                "occupation_status"
            ]
            .dropna()
            .unique()
            .tolist()
        ),


    "product_type_values":
        sorted(
            df[
                "product_type"
            ]
            .dropna()
            .unique()
            .tolist()
        ),


    "loan_intent_values":
        sorted(
            df[
                "loan_intent"
            ]
            .dropna()
            .unique()
            .tolist()
        ),


    # ---------------------------------------------------------
    # THRESHOLDS
    # ---------------------------------------------------------

    "best_threshold":
        float(
            best_threshold
        ),

    "t1":
        float(
            best_t1
        ),

    "t2":
        float(
            best_t2
        ),


    # ---------------------------------------------------------
    # MODEL PARAMETERS
    # ---------------------------------------------------------

    "best_parameters": {

        "n_estimators":
            int(
                best_params[
                    "n_estimators"
                ]
            ),

        "learning_rate":
            float(
                best_params[
                    "learning_rate"
                ]
            ),

        "max_depth":
            int(
                best_params[
                    "max_depth"
                ]
            ),

        "min_child_weight":
            int(
                best_params[
                    "min_child_weight"
                ]
            ),

        "subsample":
            float(
                fixed_params[
                    "subsample"
                ]
            ),

        "colsample_bytree":
            float(
                fixed_params[
                    "colsample_bytree"
                ]
            )

    },


    # ---------------------------------------------------------
    # DATASET INFORMATION
    # ---------------------------------------------------------

    "original_dataset":
        r"D:\Thesis\4-Dataset\Loan_approval_data_2025.csv",

    "corrected_dataset":
        corrected_dataset_path,

    "feature_columns":
        X.columns.tolist(),

    "target_column":
        "loan_status",

    "id_column":
        "customer_id",

    "train_rows":
        int(
            len(X_train)
        ),

    "validation_rows":
        int(
            len(X_val)
        ),

    "test_rows":
        int(
            len(X_test)
        ),


    # ---------------------------------------------------------
    # CLASS BALANCE
    # ---------------------------------------------------------

    "class_weights":
        {
            str(cls):
                float(weight)

            for cls, weight
            in zip(
                np.unique(
                    y_train
                ),
                class_weights
            )
        },


    "imbalance_ratio":
        float(
            imbalance_ratio
        ),


    # ---------------------------------------------------------
    # MODEL PERFORMANCE
    # ---------------------------------------------------------

    "train_auc":
        float(
            train_auc
        ),

    "validation_auc":
        float(
            val_auc
        ),

    "test_auc":
        float(
            test_auc
        ),

    "test_accuracy":
        float(
            test_accuracy
        ),

    "test_precision":
        float(
            test_precision
        ),

    "test_recall":
        float(
            test_recall
        ),

    "test_f1":
        float(
            test_f1
        ),


    "test_confusion_matrix":
        {

            "tn":
                int(tn),

            "fp":
                int(fp),

            "fn":
                int(fn),

            "tp":
                int(tp)

        },


    "test_cost":
        float(
            test_cost
        ),


    "test_fp_fn_ratio":
        float(
            fp_fn_ratio
        ),


    # ---------------------------------------------------------
    # HYPERPARAMETER SEARCH
    # ---------------------------------------------------------

    "hyperparameter_results":
        results_df,


    # ---------------------------------------------------------
    # FEATURE IMPORTANCE
    # ---------------------------------------------------------

    "feature_importance":
        pd.DataFrame(
            {
                "feature":
                    final_features,

                "importance":
                    final_model.feature_importances_
            }
        ).sort_values(
            "importance",
            ascending=False
        ),


    # ---------------------------------------------------------
    # SELECTIVE PREDICTION
    # ---------------------------------------------------------

    "selective_prediction":
        {

            "best_t1":
                float(
                    best_t1
                ),

            "best_t2":
                float(
                    best_t2
                ),

            "metrics":
                selective_metrics,

            "test_auto_accept":
                int(
                    auto_accept_mask.sum()
                ),

            "test_auto_reject":
                int(
                    auto_reject_mask.sum()
                ),

            "test_manual_review":
                int(
                    manual_review_mask.sum()
                ),

            "test_manual_review_rate":
                float(
                    manual_review_mask.mean()
                ),

            "automated_decisions":
                int(
                    auto_mask.sum()
                ),

            "selective_tn":
                int(
                    tn_auto
                ),

            "selective_fp":
                int(
                    fp_auto
                ),

            "selective_fn":
                int(
                    fn_auto
                ),

            "selective_tp":
                int(
                    tp_auto
                ),

            "selective_cost":
                float(
                    selective_cost
                ),

            "selective_fp_fn_ratio":
                float(
                    selective_fp_fn_ratio
                ),

            "manual_review_default_rate":
                float(
                    manual_review_actual_default_rate
                ),

            "manual_review_probability_mean":
                float(
                    manual_review_probability_mean
                ),

            "manual_review_probability_min":
                float(
                    manual_review_probability_min
                ),

            "manual_review_probability_max":
                float(
                    manual_review_probability_max
                )

        },


    # ---------------------------------------------------------
    # RELIABILITY MONITORING REFERENCE
    # ---------------------------------------------------------

    "reference_features":
        reference_stats,

    "reference_prediction":
        reference_pred_stats,

    "reference_performance":
        reference_performance,


    # ---------------------------------------------------------
    # MONITORING RESULT
    # ---------------------------------------------------------

    "latest_monitoring_result":
        monitoring_result,

    "monitoring_log_path":
        MONITORING_LOG_PATH,


    # ---------------------------------------------------------
    # DATA QUALITY
    # ---------------------------------------------------------

    "corrupted_rows_count":
        int(
            len(corrupted_rows)
        )

}


# =========================================================================
# 50. SAVE COMPLETE PKL
# =========================================================================

joblib.dump(
    credit_risk_package,
    PKL_PATH
)


# =========================================================================
# 51. SAVE METADATA PKL
# =========================================================================

metadata_summary = {

    "model_file":
        PKL_PATH,

    "best_threshold":
        float(
            best_threshold
        ),

    "t1":
        float(
            best_t1
        ),

    "t2":
        float(
            best_t2
        ),

    "train_auc":
        float(
            train_auc
        ),

    "validation_auc":
        float(
            val_auc
        ),

    "test_auc":
        float(
            test_auc
        ),

    "test_accuracy":
        float(
            test_accuracy
        ),

    "test_precision":
        float(
            test_precision
        ),

    "test_recall":
        float(
            test_recall
        ),

    "test_f1":
        float(
            test_f1
        ),

    "test_cost":
        float(
            test_cost
        ),

    "safe_zone":
        f"Probability < {best_t1:.2f}",

    "human_review_zone":
        (
            f"{best_t1:.2f} <= "
            f"Probability < {best_t2:.2f}"
        ),

    "risky_zone":
        f"Probability >= {best_t2:.2f}",

    "final_features":
        final_features,

    "occupation_values":
        sorted(
            df[
                "occupation_status"
            ]
            .dropna()
            .unique()
            .tolist()
        ),

    "product_type_values":
        sorted(
            df[
                "product_type"
            ]
            .dropna()
            .unique()
            .tolist()
        ),

    "loan_intent_values":
        sorted(
            df[
                "loan_intent"
            ]
            .dropna()
            .unique()
            .tolist()
        )

}


joblib.dump(
    metadata_summary,
    METADATA_PATH
)

[0]	validation_0-logloss:0.65169	validation_1-logloss:0.65124
[50]	validation_0-logloss:0.28335	validation_1-logloss:0.29370
[100]	validation_0-logloss:0.23164	validation_1-logloss:0.24658
[150]	validation_0-logloss:0.21215	validation_1-logloss:0.23029
[200]	validation_0-logloss:0.20154	validation_1-logloss:0.22194
[250]	validation_0-logloss:0.19457	validation_1-logloss:0.21746
[300]	validation_0-logloss:0.18955	validation_1-logloss:0.21485
[350]	validation_0-logloss:0.18580	validation_1-logloss:0.21404
[400]	validation_0-logloss:0.18266	validation_1-logloss:0.21340
[450]	validation_0-logloss:0.17996	validation_1-logloss:0.21336
[451]	validation_0-logloss:0.17992	validation_1-logloss:0.21340


['D:\\Thesis\\8-PKL-file\\credit_risk_metadata.pkl']